# Train model using the fastai library

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

## Setup

### Import packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import scipy.io.wavfile
import torch
from fastai.vision import *
from fastai.metrics import error_rate

import IPython.display as ipd
from tqdm import tqdm_notebook

### Instantiate ImageDataBunch

In [ ]:
pat = r'/(\w+)_\d+.jpg'

In [ ]:
from pathlib import Path
import numpy as np

path_img = "../mfc_dataset_v2"
fnames = list(Path(path_img).rglob('*.jpg')) + list(Path(path_img).rglob('*.jpeg')) + list(Path(path_img).rglob('*.png'))
np.random.seed(42)
np.random.shuffle(fnames)
fnames[:5]


In [ ]:
from fastai.vision.all import *

# ... (previous code remains unchanged)

# Use DataBlock and DataLoaders instead of ImageDataBunch
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=RegexLabeller(pat)
)

dls = dblock.dataloaders(path_img, bs=64)

# Show batch and classes
dls.show_batch()
print(dls.vocab)

# You can use dls to train your model with cnn_learner, lr_find, fit_one_cycle, etc.


For more information on batch size selection, see https://forums.fast.ai/t/how-to-check-your-pytorch-keras-is-using-the-gpu/7232/3.

In [ ]:
# Replace the old show_batch code with the following
dls.show_batch(nrows=3, ncols=3, figsize=(7,6))


In [ ]:
print(dls.vocab)
print(len(dls.vocab), dls.c)


## Train VGG16

### Transfer learning step 1: only re-train the classifier

Instantiate a learner using data, model and metric:

In [ ]:
# Tạo và huấn luyện mô hình VGG16
learn = cnn_learner(dls, models.vgg16_bn, metrics=error_rate)


**Note**: Since the model is already trained, you can now directly jump to "Evaluate model" to avoid retraining the model.

Verify the availability of GPU(s):

In [ ]:

# Kiểm tra tình trạng GPU
torch.cuda.is_available()


Find a good learning rate: (https://docs.fast.ai/callbacks.lr_finder.html)

In [ ]:

# Tìm learning rate tốt
learn.lr_find()
learn.recorder.plot_lr_find()


In [ ]:

# Huấn luyện mô hình
learn.fit_one_cycle(2, 2e-3)


In [ ]:

# Xuất mô hình sau bước 1
learn.export('vgg_model_step1.pkl')


Our model is pretty accurate after only 2 epochs.

### Transfer learning step 2: re-train the entire model

After retraining the classifier, let's also train the convolutional layers.

In [ ]:
learn.unfreeze()

In [ ]:
learn.fit_one_cycle(1)

<div style="text-align: justify">
In the context of this dataset, unfreezing and retraining the entire model sometimes do not increase its performance - but sometimes it does. For this reason, I left it here. 
</div>

In [ ]:

# Xuất mô hình sau bước 2
learn.export('vgg_model_step2.pkl')